In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD


In [2]:
#loading the Review dataset
df=pd.read_csv(r"C:\Users\HP\Desktop\Bookify\Database\Cleaned_Datasets\Ratings_Cleaned.csv")

In [3]:
df.head()

,user_id,book_id,ratings
0,276726,0155061224,5
1,276729,052165615X,3
2,276729,0521795028,6
3,276736,3257224281,8
4,276737,0600570967,6


### 1. Prepare the Interaction Matrix

In [13]:
# Limit to the first 30 rows
df_100 = df.head(100)

In [14]:
# Create the user-item interaction matrix
interaction_matrix = df_100.pivot_table(index='user_id', columns='book_id', values='ratings',  fill_value=0)


print(f"Interaction matrix shape: {interaction_matrix.shape}")
interaction_matrix.head()

Interaction matrix shape: (38, 100)


book_id,0006379702,000651118X,0060096195,0060517794,0091830893,0140260498,0141310340,0142302198,0155061224,0156006065,...,8478442588,8478884831,8478885218,8478885463,8478886044,8484330478,8484332039,8879839993,9057868059,N3453124715
user_id,,,,,,,,,,,,,,,,,,,,,
276726,0,0,0,0,0,0,0,0,5,0,...,0,0,0,0,0,0,0,0,0,0
276729,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
276736,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
276737,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
276744,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 2. User-User Collaborative Filtering

In [15]:
# Create a User-Item matrix
interaction_matrix_filled = interaction_matrix.fillna(0)  # Fill missing values with 0

# Prepare the feature matrix for KNNClassifier
X = interaction_matrix_filled.values

# Train a NearestNeighbors model
knn_model = NearestNeighbors(metric='cosine', n_neighbors=6)  
knn_model.fit(X)  


NearestNeighbors(metric='cosine', n_neighbors=6)

In [16]:
# Function to get User-User Recommendations
def user_user_knn_recommendations(user_id, top_n=5, k=5):
    user_idx = interaction_matrix_filled.index.get_loc(user_id)

    # Predict similar users using the classifier
    distances, indices = knn_model.kneighbors([X[user_idx]], n_neighbors=k+1)

    similar_users = interaction_matrix_filled.index[indices.flatten()[1:]]
    
    #  Collect books liked (rated > 3) by similar users
    recommended_books = []
    for sim_user in similar_users:
        liked_books = interaction_matrix.loc[sim_user][interaction_matrix.loc[sim_user] > 3].index.tolist()
        recommended_books.extend(liked_books)

    # Remove books the target user has already rated
    already_rated = interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 0].index.tolist()
    final_books = [book for book in recommended_books if book not in already_rated]

    # If no books are left after filtering, return empty list
    if not final_books:
        return []

    # Step 4.5: Recommend top N books (most frequently liked)
    return pd.Series(final_books).value_counts().head(top_n).index.tolist()

# Test for a few users
for user in df['user_id'].unique()[:5]:
    print(f"User-User Recommendations for User {user}: {user_user_knn_recommendations(user)}")


User-User Recommendations for User 276726: ['8879839993', '0091830893', '0586207414', '0812571029', '8423996565']
User-User Recommendations for User 276729: ['0395547032', '8423996565', '8426449476', '8426449573', '8478884831']
User-User Recommendations for User 276736: ['8879839993', '0395547032', '8423996565', '8426449476', '8426449573']
User-User Recommendations for User 276737: ['8879839993', '0395547032', '8423996565', '8426449476', '8426449573']
User-User Recommendations for User 276744: ['0440414121', '8423996565', '8426449476', '8426449573', '8478884831']


### 3. Implement Item-Item Collaborative Filtering

In [17]:
# Compute item-item cosine similarity
item_similarity = pd.DataFrame(cosine_similarity(interaction_matrix.T),
                               index=interaction_matrix.columns,
                               columns=interaction_matrix.columns)


In [18]:
def item_item_recommendations(user_id, top_n=5, sim_n=5):
    
    # Get books rated > 3 by the user
    liked_books = interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 3].index
    recommendations = []

    
    for book in liked_books:
        similar_books = item_similarity[book].sort_values(ascending=False)
        top_similar_books = similar_books.iloc[1:sim_n+1].index 
        for similar_book in top_similar_books:
            if similar_book not in liked_books:
                recommendations.append(similar_book)

    # Return top recommended books based on frequency
    return pd.Series(recommendations).value_counts().head(top_n).index.tolist()

# Test on 5 users
for user in interaction_matrix.index[:5]:
    print(f"User {user} => Item-Item Recommendations: {item_item_recommendations(user)}")

User 276726 => Item-Item Recommendations: ['0006379702', '3442422035', '3453213025', '3453137442', '3453092007']
User 276729 => Item-Item Recommendations: ['3442435773', '347354034X', '3453213025', '3453137442']
User 276736 => Item-Item Recommendations: ['3442422035', '3453213025', '3453137442', '3453092007', '3442449820']
User 276737 => Item-Item Recommendations: ['0006379702', '3442435773', '3453213025', '3453137442', '3453092007']
User 276744 => Item-Item Recommendations: ['0006379702', '3442435773', '3453213025', '3453137442', '3453092007']


### 4. Matrix Factorization

In [10]:
#Apply SVD to reduce dimensions (20 features)
svd = TruncatedSVD(n_components=20, random_state=42)
compressed_matrix = svd.fit_transform(interaction_matrix)

#Rebuild the matrix (predict missing ratings)
reconstructed_matrix = np.dot(compressed_matrix, svd.components_)
predicted_ratings_df = pd.DataFrame(reconstructed_matrix, 
                                    index=interaction_matrix.index, 
                                    columns=interaction_matrix.columns)


In [11]:
#Recommend books for a user based on predicted ratings
def svd_recommendations(user_id, top_n=5):
    # Books the user has already rated
    rated_books = interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 0].index
    
    # Get predicted ratings for books the user hasn't rated
    predictions = predicted_ratings_df.loc[user_id].drop(rated_books)
    
    # Return top N recommended book IDs
    return predictions.sort_values(ascending=False).head(top_n).index.tolist()

for user in interaction_matrix.index[:5]:
    recommendations = svd_recommendations(user)
    print(f"User {user} => SVD Recommendations: {recommendations}")


User 276726 => SVD Recommendations: ['0671537458', '0060517794', '3499230933', '3596151465', '0679776818']
User 276729 => SVD Recommendations: ['3596218098', '0395547032', '8484332039', '8484330478', '342310538']
User 276736 => SVD Recommendations: ['3442136644', '0747558167', '0684867621', '0440414121', '3596218098']
User 276737 => SVD Recommendations: ['8879839993', '0440414121', '8484332039', '8484330478', '055310666X']
User 276744 => SVD Recommendations: ['0330332775', '8879839993', '1562827898', '8440682697', '0671537458']


### 5. Evaluation & Observations

In [12]:
def compare_models(user_id):
    print(f"\nRecommendations for User {user_id}")
    print("Books already rated:", list(interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 0].index))
    print("User-User CF Recommendations:", user_user_knn_recommendations(user_id))
    print("Item-Item CF Recommendations:", item_item_recommendations(user_id))
    print("SVD Recommendations:", svd_recommendations(user_id))
    print()

# Evaluate a few users
for uid in interaction_matrix.index[:3]:
    compare_models(uid)



Recommendations for User 276726
Books already rated: ['0155061224']
User-User CF Recommendations: ['0345443683', '043935806X', '055310666X', '8484330478', '8484332039']
Item-Item CF Recommendations: ['0006379702', '8437606322', '3442136644', '3453092007', '3453213025']
SVD Recommendations: ['0671537458', '0060517794', '3499230933', '3596151465', '0679776818']


Recommendations for User 276729
Books already rated: ['052165615X', '0521795028']
User-User CF Recommendations: ['0345443683', '043935806X', '055310666X', '8484330478', '8484332039']
Item-Item CF Recommendations: ['052165615X', '8437606322', '3442136644', '3453092007', '3453213025']
SVD Recommendations: ['3596218098', '0395547032', '8484332039', '8484330478', '342310538']


Recommendations for User 276736
Books already rated: ['3257224281']
User-User CF Recommendations: ['0345443683', '043935806X', '055310666X', '0006379702', '3442131340']
Item-Item CF Recommendations: ['8426449573', '3442131340', '3442136644', '3453092007', '3